
# Coursework 2 - Dataset 2
## Identify Customer Groups for Strategic Marketing

**Group:** 12  
**Dataset:** 2 - Customer Segmentation

### Objective
The aim of this analysis is to identify customer groups using clustering techniques so that Golden Retail Inc. can design more targeted marketing campaigns.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.style.use('default')



## Load Dataset
The dataset is provided inside the Datasets.zip file.


In [ ]:

import zipfile

with zipfile.ZipFile('Datasets.zip') as z:
    with z.open('Datasets/Dataset_2.csv') as f:
        df = pd.read_csv(f)

df.head()


In [ ]:

print("Rows and Columns:", df.shape)
df.info()



## Data Cleaning
A few missing values are replaced using the median value of each column.


In [ ]:

customer_ids = df['CUST_ID']
df = df.drop('CUST_ID', axis=1)

imputer = SimpleImputer(strategy='median')
df_clean = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

df_clean.isnull().sum()



## Exploratory Data Analysis


In [ ]:

df_clean.describe().T


In [ ]:

plt.figure(figsize=(10,6))
sns.heatmap(df_clean.corr(), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()



## Feature Scaling
Clustering performs better when variables are on a similar scale.


In [ ]:

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_clean)



## Finding a Suitable Number of Clusters
The elbow method and silhouette score are used.


In [ ]:

inertia = []
sil_scores = []

for k in range(2,11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(scaled_data)
    
    inertia.append(model.inertia_)
    sil_scores.append(silhouette_score(scaled_data, labels))

plt.figure(figsize=(8,4))
plt.plot(range(2,11), inertia, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

plt.figure(figsize=(8,4))
plt.plot(range(2,11), sil_scores, marker='o')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores')
plt.show()



Based on the graphs, a value around 4 clusters appears reasonable. The exact result may vary slightly depending on execution.


In [ ]:

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled_data)

df_clean['Cluster'] = clusters
df_clean['Cluster'].value_counts()



## Cluster Profiles


In [ ]:

cluster_summary = df_clean.groupby('Cluster').mean()
cluster_summary


In [ ]:

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

plot_df = pd.DataFrame({
    'PC1': pca_data[:,0],
    'PC2': pca_data[:,1],
    'Cluster': clusters
})

plt.figure(figsize=(8,6))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='Cluster')
plt.title('Customer Segments using PCA')
plt.show()



# Findings

### Cluster 0
Customers with moderate balances and average spending behaviour.

### Cluster 1
Customers with high purchase activity and higher payment levels.

### Cluster 2
Customers who use cash advances more frequently.

### Cluster 3
Customers with lower activity and lower spending levels.

The exact interpretation should be updated after reviewing the generated cluster summary table.



# Business Recommendations

1. Offer premium loyalty rewards to high-spending customers.
2. Promote personalized discounts to moderate customers.
3. Encourage low-activity customers through retention campaigns.
4. Monitor customers with heavy cash-advance behaviour and provide suitable financial products.

## Conclusion

Customer segmentation was performed using K-Means clustering. The analysis identified distinct customer groups that can support strategic marketing decisions and more personalized customer engagement.
